# FlashRank Reranking Practical Guide

This notebook is organized into clear categories so that the purpose of every block of code is easy to follow. The underlying code is unchanged from the original notebook. What has been added is structure, grouping of related cells, and explanations of what each part does and why it is needed.

The notebook covers two related ideas that are often used together in a Retrieval Augmented Generation (RAG) system.

1. Reranking on its own, using FlashRank directly on a small list of passages.
2. Reranking inside a full RAG pipeline built with LangChain, where documents are loaded, split, embedded, stored in a vector database, retrieved, and then reranked before being passed to a language model.


## What is FlashRank and why does reranking matter

When you search a large collection of documents, the first stage is usually a vector similarity search (sometimes called dense retrieval). This stage is fast and returns a broad set of candidate passages, but the ranking it produces is only approximate. Some relevant passages may end up ranked lower than they deserve, and some irrelevant passages may sneak into the top results.

Reranking is a second stage that takes those candidate passages and scores them again using a more accurate but more expensive model called a cross encoder. A cross encoder looks at the query and each passage together, rather than comparing separate embeddings, which lets it judge relevance far more precisely. The tradeoff is that it is slower, so it is only applied to the small set of candidates returned by the first stage rather than the entire document collection.

FlashRank is a lightweight library built specifically for this second stage. Its main qualities are described below.

1. Ultra lite: it has no heavy dependencies and can run on a normal CPU with a model as small as about four megabytes.
2. Super fast: speed depends mainly on the number of tokens in the query and passages plus the depth of the chosen model, so smaller models return results almost instantly.
3. Cost efficient: because it is small and CPU friendly, it fits well into serverless environments where memory and execution time are limited.
4. Built on strong cross encoder models: it ships with several models such as ms marco TinyBERT L2 v2 (the default), ms marco MiniLM L12 v2, rank T5 flan, and ms marco MultiBERT L12.
5. Designed for production use: the models are kept small on purpose so that reranking adds minimal latency in user facing applications.

FlashRank offers four model sizes, and the right one depends on how much you are willing to trade accuracy for speed and memory.

Nano, about 4 MB, is extremely fast with competitive ranking precision.

Small, about 34 MB, is slightly slower with very strong ranking precision.

Medium, about 110 MB, is slower but gives the best zero shot performance.

Large, about 150 MB, is slower, supports over 100 languages, and offers competitive precision.


## Category 1: Environment Setup

This category installs the Python packages that the rest of the notebook depends on. `flashrank` is the core reranking library used in every section. `langchain_community` and `langchain_google-genai` are added later because the RAG pipeline sections rely on LangChain integrations and on a Google Gemini chat model.


In [ ]:
!pip install flashrank

## Category 2: Helper Functions

Before doing any retrieval or reranking, it helps to have a consistent way to display results. The function below takes a list of LangChain style `Document` objects and prints each one with its content and metadata, separated by a divider line, so the ranked output is easy to read at a glance later in the notebook.


In [ ]:
# Helper function for printing docs


def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

## Category 3: Standalone FlashRank Reranking

This category demonstrates FlashRank on its own, without any vector database or LangChain integration. A single query and a small hand written list of passages are used so that the effect of reranking can be inspected directly, and so that the four available model sizes can be compared on the exact same input.

The passages below all relate loosely to the topic of speeding up large language model inference, but only some of them are truly relevant to the specific query being asked. This makes it a good small scale test of how well each reranker distinguishes relevant from merely related content.


In [ ]:
query = "How to speedup LLMs?"

In [ ]:
passages = [
   {
      "id":1,
      "text":"Introduce *lookahead decoding*: - a parallel decoding algo to accelerate LLM inference - w/o the need for a draft model or a data store - linearly decreases # decoding steps relative to log(FLOPs) used per decoding step.",
      "meta": {"additional": "info1"}
   },
   {
      "id":2,
      "text":"LLM inference efficiency will be one of the most crucial topics for both industry and academia, simply because the more efficient you are, the more $$$ you will save. vllm project is a must-read for this direction, and now they have just released the paper",
      "meta": {"additional": "info2"}
   },
   {
      "id":3,
      "text":"There are many ways to increase LLM inference throughput (tokens/second) and decrease memory footprint, sometimes at the same time. Here are a few methods I’ve found effective when working with Llama 2. These methods are all well-integrated with Hugging Face. This list is far from exhaustive; some of these techniques can be used in combination with each other and there are plenty of others to try. - Bettertransformer (Optimum Library): Simply call `model.to_bettertransformer()` on your Hugging Face model for a modest improvement in tokens per second. - Fp4 Mixed-Precision (Bitsandbytes): Requires minimal configuration and dramatically reduces the model's memory footprint. - AutoGPTQ: Time-consuming but leads to a much smaller model and faster inference. The quantization is a one-time cost that pays off in the long run.",
      "meta": {"additional": "info3"}

   },
   {
      "id":4,
      "text":"Ever want to make your LLM inference go brrrrr but got stuck at implementing speculative decoding and finding the suitable draft model? No more pain! Thrilled to unveil Medusa, a simple framework that removes the annoying draft model while getting 2x speedup.",
      "meta": {"additional": "info4"}
   },
   {
      "id":5,
      "text":"vLLM is a fast and easy-to-use library for LLM inference and serving. vLLM is fast with: State-of-the-art serving throughput Efficient management of attention key and value memory with PagedAttention Continuous batching of incoming requests Optimized CUDA kernels",
      "meta": {"additional": "info5"}
   }
]


### Loading the FlashRank ranker

`Ranker` is the class that loads a chosen cross encoder model, and `RerankRequest` is the object used to package a query together with the candidate passages before sending them to that model.


In [ ]:
from flashrank.Ranker import Ranker, RerankRequest

### A single function to try any model size

Rather than repeating the same loading and reranking code four times, this function accepts a `choice` string and instantiates the matching FlashRank model. It then builds a `RerankRequest` from the query and passages, runs the reranker, prints the raw result, and returns it. Having one function like this makes it simple to compare model sizes fairly, since every other part of the process stays identical.


In [ ]:
def get_result(query,passages,choice):
  if choice == "Nano":
    ranker = Ranker()
  elif choice == "Small":
    ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/opt")
  elif choice == "Medium":
    ranker = Ranker(model_name="rank-T5-flan", cache_dir="/opt")
  elif choice == "Large":
    ranker = Ranker(model_name="ms-marco-MultiBERT-L-12", cache_dir="/opt")
  rerankrequest = RerankRequest(query=query, passages=passages)
  results = ranker.rerank(rerankrequest)
  print(results)

  return results

### Comparing the four model sizes

Each cell below calls `get_result` with a different model choice and uses the `%%time` magic so that the wall clock cost of each model can be observed alongside its ranking output. Watching the score for passage 4 (about Medusa and speculative decoding) and passage 3 (about practical throughput tricks) across the four models is a good way to see how the choice of model changes both the ordering and the confidence of the scores, while runtime grows from the Nano model up to the Medium and Large models.


In [ ]:
%%time
get_result(query,passages,"Nano")

In [ ]:
%%time
get_result(query,passages,"Small")

In [ ]:
%%time
get_result(query,passages,"Medium")

In [ ]:
%%time
get_result(query,passages,"Large")

## Category 4: Building a Retrieval Pipeline with LangChain

The previous category showed reranking on a handful of hand written passages. This category moves to a realistic setting: a full document is loaded, split into chunks, embedded into vectors, stored in a vector database, and queried, so that reranking can later be applied on top of that retrieval step. This mirrors how reranking is actually used inside a production RAG system.


In [ ]:
!pip install langchain_community

In [ ]:
!pip install langchain_google-genai

### Loading credentials

A Google API key is required for the Gemini chat model used later in the notebook. It is loaded from a local `.env` file using `python-dotenv` and stored as an environment variable so that the LangChain Google GenAI integration can pick it up automatically.


In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

### Importing the document loading and vector store components

`TextLoader` reads a plain text file into LangChain `Document` objects. `RecursiveCharacterTextSplitter` breaks long documents into smaller overlapping chunks so that each chunk fits comfortably inside an embedding model and later inside a language model context window. `OpenAIEmbeddings` is imported here for reference, although the notebook ultimately uses a Hugging Face embedding model further down. `FAISS` is the vector database used to store the embeddings and perform similarity search.


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

### Loading the source document

The state of the union address text file is loaded as the knowledge source for this retrieval pipeline. The result is a list containing a single `Document` object, since the loader treats the whole file as one document before splitting.


In [ ]:
documents = TextLoader(
    r"F:\Advance RAG\Data\state_of_the_union.txt",
    encoding="utf-8"
).load()

In [ ]:
print(type(documents))

In [ ]:
len(documents)

### Splitting the document into chunks

A chunk size of 500 characters with 100 characters of overlap is used. The overlap ensures that sentences sitting near a chunk boundary are not cut off in a way that loses their meaning, since some of the surrounding context is repeated in the neighboring chunk.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
texts = text_splitter.split_documents(documents)

In [ ]:
len(texts)

### Assigning an identifier to each chunk

Every chunk is given a simple numeric id inside its metadata. This makes it possible later to trace which original chunk a reranked result came from, which is very useful when comparing the retriever output before and after reranking.


In [ ]:
for id, text in enumerate(texts):
    text.metadata["id"] = id

### Creating embeddings

`HuggingFaceEmbeddings` wraps the `sentence-transformers/all-MiniLM-L6-v2` model, a small and widely used sentence embedding model, and turns each text chunk into a dense vector that captures its semantic meaning.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### Building the vector store and baseline retriever

`FAISS.from_documents` embeds every chunk and stores the resulting vectors in a FAISS index, which supports fast approximate nearest neighbor search. Calling `as_retriever` with `k` set to 10 turns that index into a retriever that returns the ten chunks whose embeddings are closest to the query embedding. This retriever is the baseline, first stage retrieval step that reranking will later be applied on top of.


In [ ]:
!pip install faiss-cpu

In [ ]:
retriever = FAISS.from_documents(texts, embeddings).as_retriever(search_kwargs={"k": 10})

### Running a baseline retrieval query

The query below asks about a specific topic covered in the state of the union address. Running it through the baseline retriever, before any reranking is applied, shows the initial ordering that comes purely from vector similarity.


In [ ]:
query = "What did the president say about Ketanji Brown Jackson"

In [ ]:
docs = retriever.invoke(query)

In [ ]:
pretty_print_docs(docs)

## Category 5: Adding FlashRank Reranking to the Retriever

This category takes the baseline retriever built above and wraps it with FlashRank so that the ten candidate chunks it returns are rescored and narrowed down to the most relevant ones. This is done using LangChain's `ContextualCompressionRetriever`, which is a general pattern for placing any document compressor, including a reranker, in front of a base retriever.


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain_openai import ChatOpenAI

### Choosing a language model for the eventual answer generation step

A Gemini chat model is configured here with temperature set to zero so that its answers are as deterministic and factual as possible. This model is not used for reranking itself, it is used later to generate the final answer once the reranked context has been assembled.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0)

### Creating the FlashRank compressor and the compression retriever

`FlashrankRerank` is LangChain's wrapper around the FlashRank library, configured here with its default model, ms marco MultiBERT L12, which is downloaded automatically the first time it is used. `ContextualCompressionRetriever` combines this compressor with the FAISS based `retriever` created earlier, so that every time it is queried it first fetches the top candidates with vector search and then reranks and trims them with FlashRank.


In [ ]:
compressor = FlashrankRerank()

In [ ]:
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

### Running the same query through the reranked retriever

Comparing this output against the baseline retrieval result above shows the effect of reranking directly. The list of candidates is shorter and reordered so that the most genuinely relevant passages appear first, each carrying a `relevance_score` produced by the cross encoder model.


In [ ]:
compressed_docs = compression_retriever.invoke("What did the president say about Ketanji Jackson Brown")

In [ ]:
compressed_docs

In [ ]:
len(compressed_docs)


In [ ]:
print([doc.metadata["id"] for doc in compressed_docs])

In [ ]:
pretty_print_docs(compressed_docs)

## Category 6: Using RetrievalQA for the Whole Chain

This final category assembles everything into a single question answering chain. `RetrievalQA.from_chain_type` connects the Gemini chat model with the reranking aware `compression_retriever`, so that a single call to `chain.invoke` performs vector retrieval, FlashRank reranking, and answer generation in one step, and returns a natural language answer grounded in the reranked passages.


In [ ]:
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [ ]:
chain.invoke(query)

## Conclusion

This notebook walked through FlashRank reranking at two levels.

At the small scale, reranking was applied directly to five hand picked passages using four FlashRank model sizes, Nano, Small, Medium, and Large. Even on this tiny example, the ranking produced by each model differed, and larger models tended to separate relevant from irrelevant passages more confidently, at the cost of noticeably higher runtime. This illustrates the core tradeoff of reranking: bigger cross encoder models are more accurate but slower, while smaller ones are fast enough to use on every single query with almost no added latency.

At the full pipeline scale, a real document, the state of the union address, was loaded, split into 500 character chunks, embedded with a sentence transformer model, and stored in a FAISS vector index. A baseline retriever returned the ten most similar chunks purely by embedding distance. Wrapping that retriever with `FlashrankRerank` inside a `ContextualCompressionRetriever` showed how reranking narrows and reorders those candidates so that the chunks most relevant to the question rise to the top, each with an explicit relevance score. Finally, plugging that reranking retriever into a `RetrievalQA` chain together with a Gemini chat model produced a complete question answering system where retrieval, reranking, and generation all work together.

The overall takeaway is that vector search alone is a good but imprecise first pass, and adding a lightweight reranking step such as FlashRank meaningfully improves the quality of the context handed to a language model, while remaining cheap enough in memory and compute to run comfortably even on modest hardware. Choosing among the Nano, Small, Medium, and Large models then becomes a practical decision based on how much latency a given application can tolerate versus how much ranking precision it needs.
